# SENTINEL on the official harness, with the real Qwen3-8B agent

This notebook runs the organizers' evaluator against **our defense service** while the agent
being defended is the official reference agent — `Qwen/Qwen3-8B` — instead of the deterministic
mock. That is the difference between "our decision logic scores well on scripted plans" and
"our decision logic holds up when a real model is the thing proposing actions".

**Kaggle settings (right-hand panel):**

| setting | value |
|---|---|
| Accelerator | **GPU T4 x2** (two 16 GB cards — 16-bit Qwen3-8B does not fit on one) |
| Internet | **On** (clones the repos, downloads the weights) |
| Add-ons ▸ Secrets | `GITHUB_TOKEN` — **only while the defense repository is private** (a fine-grained token with read access to that one repo). Alternatively upload the repository as a Kaggle Dataset and attach it. |
| Persistence | Variables and files: off is fine; results are zipped at the end |

A single T4 or P100 also works if you set `PRECISION = "4bit"` in the config cell.

**What it does, in order:** install → clone the kit and the defense → fetch the weights →
start the defense service → check the attack actually lands with no defense → run the four
evaluations → write the table and the failure list for the report.

**Budget:** expect ~20–40 min per evaluation split, so ~2–3 h for all four, inside Kaggle's
12 h session and 30 h/week GPU quota. Run the first split, look at the clock, then decide.

## 0 · What hardware did we actually get

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

print(sys.version)
if sys.version_info < (3, 11):
    raise SystemExit("the starter kit uses enum.StrEnum and datetime.UTC: Python 3.11+ is required")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "no GPU")
except FileNotFoundError:
    print("no nvidia-smi: this notebook needs a GPU accelerator")

for path in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    if Path(path).exists():
        free = shutil.disk_usage(path).free / 1e9
        print(f"{path:16s} {free:6.1f} GB free")

## 1 · Configuration

The only cell you should need to edit.

In [ ]:
DEFENSE_REPO = "https://github.com/wissemkooli/sentinel-indabax.git"
DEFENSE_REF  = "main"
KIT_REPO     = "https://github.com/Skan22/Sentinel_Starter_Kit.git"

PRECISION    = "fp16"     # fp16 on 2x T4 (faithful 16-bit) | 4bit or 8bit on a single card
DEFENSE_PORT = 8099

# (split, attacker, attack_mode). The first pair is the headline number; the mutation/adaptive
# pair is the one that shows the defense is not overfitted to a fixed set of injected strings.
RUNS = [
    ("public",     "static",   "static"),
    ("validation", "static",   "static"),
    ("public",     "mutation", "adaptive"),
    ("validation", "mutation", "adaptive"),
]

# Each pass runs every scenario in the split exactly once, at this run seed. The seed shuffles
# the synthetic world and the attacker's choices, so extra seeds are extra evidence -- and extra
# GPU hours: one more seed is one more full pass. Decoding is greedy, so re-running the same seed
# reproduces the same transcript and tells you nothing new.
SEEDS = [0]

# The control: the same agent with no defense, over both splits. Not optional. Qwen3-8B fails
# some tasks and ignores some injections all by itself, and without this pass nobody can tell
# a false block from an agent failure, or a contained attack from one that never happened.
RUN_CONTROL = True

RUN_BASELINES = False     # also score the kit's provenance / heuristic_risk under Qwen3-8B
                          # (one extra pass over the public split each -- check your quota first)

import os, shutil
from pathlib import Path

KAGGLE  = Path("/kaggle").exists()
WORK    = Path("/kaggle/working") if KAGGLE else Path.cwd() / "sentinel-qwen3"
SCRATCH = max((Path(p) for p in ("/kaggle/temp", "/tmp", str(WORK)) if Path(p).exists()),
              key=lambda p: shutil.disk_usage(p).free)
KIT      = WORK / "Sentinel_Starter_Kit"
REPO     = WORK / "sentinel-indabax"
RESULTS  = WORK / "results"
ARTIFACTS = WORK / "artifacts"
TRACES   = WORK / "traces"       # our observability layer's recordings, one JSONL per run
for d in (WORK, RESULTS, ARTIFACTS, TRACES):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(SCRATCH / "hf")
os.environ["SENTINEL_QWEN_PRECISION"] = PRECISION
print(f"work={WORK}\nweights cache={os.environ['HF_HOME']} ({shutil.disk_usage(SCRATCH).free/1e9:.0f} GB free)")

## 2 · Dependencies

Kaggle ships torch; the rest is the kit's runtime plus the defense service's.

In [ ]:
packages = [
    "transformers>=4.51",   # Qwen3 support landed in 4.51
    "accelerate>=0.30",     # device_map sharding across the two T4s
    "typer>=0.12", "rich>=13.7", "httpx>=0.27",
    "pydantic>=2.8,<3", "fastapi>=0.115", "uvicorn>=0.30", "pyyaml>=6.0.2",
    "kagglehub",
]
if PRECISION in ("4bit", "8bit"):
    packages.append("bitsandbytes>=0.43")

spec = " ".join(f'"{p}"' for p in packages)
!pip install -q {spec}

import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda devices", torch.cuda.device_count())
if torch.cuda.device_count() < 2 and PRECISION in ("fp16", "bf16"):
    print("\nWARNING: one GPU and 16-bit weights will not fit. Set PRECISION = '4bit' and rerun cell 1.")

## 3 · The kit and the defense

In [ ]:
import subprocess, sys

def github_token():
    """A private defense repository needs a token; a public one does not."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

def attached_copy(marker):
    """The repository uploaded as a Kaggle Dataset and attached as an input."""
    root = Path("/kaggle/input")
    hits = sorted(root.rglob(marker)) if root.exists() else []
    return hits[0].parents[len(Path(marker).parts) - 1] if hits else None

def clone(url, dest, ref=None, token=None, marker=None):
    if dest.exists():
        print(f"{dest.name}: already present")
        return
    authed = url.replace("https://", f"https://x-access-token:{token}@") if token else url
    cmd = ["git", "clone", "--depth", "1"] + (["--branch", ref] if ref else []) + [authed, str(dest)]
    done = subprocess.run(cmd, capture_output=True, text=True)
    if done.returncode == 0:
        subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", url])  # keep the token out of .git
        return
    copy = attached_copy(marker) if marker else None
    if copy:
        shutil.copytree(copy, dest)
        print(f"{dest.name}: copied from attached input {copy}")
        return
    raise SystemExit(
        f"could not clone {url}\n{done.stderr.replace(token or chr(0), '***')[-400:]}\n"
        "If the repository is private: add a GITHUB_TOKEN secret (Add-ons > Secrets) with read access, "
        "or upload the repository as a Kaggle Dataset and attach it to this notebook.")

clone(KIT_REPO, KIT)
clone(DEFENSE_REPO, REPO, DEFENSE_REF, token=github_token(), marker="sentinel/api_adapter.py")

# The kit declares python >=3.12,<3.13 and Kaggle may be on 3.11; nothing in it needs 3.12, so
# import it from source instead of installing the package metadata.
# Only the kit and our kaggle/ helpers go on this process's path. The kit's package and our defense
# package are both called `sentinel`, so our own tools run in subprocesses (see `ours` below).
sys.path[:0] = [str(KIT / "src"), str(REPO / "kaggle")]

def ours(*argv):
    """Run one of our repository's tools with our `sentinel` package on the path, not the kit's."""
    done = subprocess.run([sys.executable, *argv], cwd=str(REPO), env=dict(os.environ, PYTHONPATH=str(REPO)),
                          capture_output=True, text=True)
    print(done.stdout + done.stderr[-2000:])
    return done.returncode
os.environ["SENTINEL_ROOT"] = str(KIT)   # where policies/ and fixtures/ live

for repo in (KIT, REPO):
    head = subprocess.run(["git", "-C", str(repo), "log", "-1", "--format=%h %s"],
                          capture_output=True, text=True).stdout.strip()
    print(f"{repo.name:24s} {head or '(no git history: copied from an input)'}")

if not (REPO / "observability" / "live.py").exists():
    raise SystemExit("this checkout predates the live trace recorder: push the latest commit, "
                     "or re-upload the dataset, then restart the session")

## 4 · Qwen3-8B weights

Tried in order: a Kaggle Model attached as a notebook input, then Kaggle Models, then the
Hugging Face hub. ~16 GB either way, so this is the slow cell on a cold start.

In [ ]:
import json

def looks_like_qwen3_8b(config: Path) -> bool:
    try:
        data = json.loads(config.read_text())
    except Exception:
        return False
    return data.get("model_type", "").startswith("qwen3") and data.get("num_hidden_layers") == 36

def from_inputs():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for config in sorted(root.rglob("config.json")):
        if looks_like_qwen3_8b(config):
            return config.parent

def from_kagglehub():
    import kagglehub
    for handle in ("qwen-lm/qwen-3/transformers/8b", "qwen-lm/qwen-3/transformers/qwen3-8b"):
        try:
            return Path(kagglehub.model_download(handle))
        except Exception as exc:
            print(f"  {handle}: {type(exc).__name__}")

def from_huggingface():
    from huggingface_hub import snapshot_download
    return Path(snapshot_download("Qwen/Qwen3-8B"))

WEIGHTS = os.environ.get("SENTINEL_WEIGHTS")
if not WEIGHTS:
    for source in (from_inputs, from_kagglehub, from_huggingface):
        print(f"trying {source.__name__} ...")
        found = source()
        if found:
            WEIGHTS = str(found)
            break
if not WEIGHTS:
    raise SystemExit("no Qwen3-8B weights: attach the Kaggle model as an input, or enable internet")

size = sum(f.stat().st_size for f in Path(WEIGHTS).rglob("*") if f.is_file()) / 1e9
print(f"\nweights: {WEIGHTS}  ({size:.1f} GB)")

## 5 · Start the defense service

The same FastAPI app the submission container runs, on the same `/v1/decision` contract. The
evaluator reaches it over HTTP exactly as the organizers' runner would.

It runs with `SENTINEL_TRACE_DIR` set, so every run below is also recorded in our own trace
format — mandate, provenance, evidence, the log-odds arithmetic — for `sentinel replay` and the
HTML dashboard. Recording never feeds back into a decision.

In [ ]:
import time, urllib.request, urllib.error

DEFENSE_URL = f"http://127.0.0.1:{DEFENSE_PORT}"
log_path = RESULTS / "defense-service.log"

def healthz():
    try:
        with urllib.request.urlopen(f"{DEFENSE_URL}/healthz", timeout=2) as response:
            return json.loads(response.read())
    except (urllib.error.URLError, OSError, json.JSONDecodeError):
        return None

def start_defense():
    """Start the service if it is not answering. Called before every run, because a service that
    died mid-session makes the harness fail closed and every action shows up as DEFENSE_UNAVAILABLE."""
    global defense
    if healthz():
        return
    env = dict(os.environ, PYTHONPATH=f"{REPO}:{REPO / 'submission'}",
               SENTINEL_TRACE_DIR=str(TRACES),                       # record every run for the dashboard
               SENTINEL_AGENT_LABEL=f"Qwen/Qwen3-8B ({PRECISION}, kit prompt, thinking off)")
    for proxy in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        env.pop(proxy, None)
    defense = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", str(DEFENSE_PORT),
         "--log-level", "info"],
        cwd=str(REPO / "submission"), env=env,
        stdout=log_path.open("a"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if healthz():
            return
        if defense.poll() is not None:
            break
        time.sleep(1)
    print(log_path.read_text()[-3000:])
    raise SystemExit("the defense service did not come up; its log is above")

def probe_decision():
    """One real POST through the same client the harness uses, so a failure is named here rather
    than showing up later as DEFENSE_UNAVAILABLE on every step."""
    import httpx
    body = {"run_id": "probe", "step_id": 1, "user_goal": "Read document DOC-1 and summarize it.",
            "conversation": [], "candidate_action": {"type": "tool_call", "tool": "document_read",
                                                     "arguments": {"doc_id": "DOC-1"}},
            "policy_context": {"allowed_tools": ["document_read"]}, "provenance": [],
            "history_digest": {}}
    try:
        reply = httpx.post(f"{DEFENSE_URL}/v1/decision", json=body, timeout=5.0)
        print("probe:", reply.status_code, reply.text[:300])
        return reply.status_code == 200
    except Exception as exc:
        print(f"probe failed: {type(exc).__name__}: {exc}")
        print(log_path.read_text()[-3000:])
        return False

# The harness reaches the service through httpx, which honours proxy variables; a proxy cannot
# reach 127.0.0.1 on this machine.
os.environ["NO_PROXY"] = os.environ["no_proxy"] = "127.0.0.1,localhost"

start_defense()
print("defense:", healthz())
if not probe_decision():
    raise SystemExit("the service is up but /v1/decision does not answer correctly; see above")


## 6 · Does the attack even reach the agent?

The kit's own warning: if an *undefended* run does not report `attack_success=True`, the agent
never opened the injected record and every later number is meaningless. So before scoring
anything, run one attack scenario with `allow_all`, then the same scenario against our defense.

This is also where the 16 GB of weights get loaded — once, and then reused for everything below.

In [ ]:
import qwen_runtime
print(qwen_runtime.apply())
from sentinel.cli import app

SCENARIO = str(KIT / "scenarios/public/finance/finance_false_approval.yaml")

def sentinel(*argv):
    """Invoke the kit's CLI in this process, so the model stays loaded between runs."""
    started = time.time()
    if "--defense-url" in argv:
        start_defense()
    try:
        app(args=list(argv), standalone_mode=False)
    except SystemExit as exc:
        if exc.code:
            raise
    print(f"[{time.time() - started:.0f}s]")

def explain_unavailable():
    """If the harness could not get a decision, say why: its own error text, then the service log."""
    runs = sorted(ARTIFACTS.rglob("*http_defense*.jsonl"), key=lambda f: f.stat().st_mtime)
    if not runs:
        return
    errors = {e["payload"].get("explanation") for e in map(json.loads, runs[-1].read_text().splitlines())
              if e.get("type") == "defense_decision" and "DEFENSE_UNAVAILABLE" in e["payload"].get("reason_codes", [])}
    if errors:
        print("\n!! The harness could not get decisions from the defense, so it blocked everything:")
        for error in errors:
            print("   ", error)
        print("--- defense-service.log (tail) ---\n" + log_path.read_text()[-3000:])

def file_traces(label):
    """Attach the harness's verdicts to the recordings of the pass that just ran and file them
    under their own directory, because the harness reuses run ids from one pass to the next."""
    ours("-m", "observability.live", "attach", str(TRACES), str(ARTIFACTS))
    dest = TRACES / label
    dest.mkdir(parents=True, exist_ok=True)
    moved = [shutil.move(str(f), str(dest / f.name)) for f in TRACES.glob("*.jsonl")]
    print(f"{len(moved)} trace(s) -> {dest}")
    explain_unavailable()
    return dest

sentinel("run", "--scenario", SCENARIO, "--defense", "allow_all", "--model", WEIGHTS,
         "--artifacts", str(ARTIFACTS))
file_traces("check-undefended")   # allow_all never calls our service, so this is usually empty


In [ ]:
sentinel("run", "--scenario", SCENARIO, "--defense-url", DEFENSE_URL, "--model", WEIGHTS,
         "--artifacts", str(ARTIFACTS))
file_traces("check-defended")


## 6b · Try any scenario against Qwen3-8B

The cell below lists the scenario library. `try_scenario(...)` runs one scenario — undefended
first if you ask for it, which is the kit's reachability check — then prints the rich trace of
our defense's decisions. This is the cell to use when choosing what to record for the video.

In [ ]:
LIBRARY = {p.stem: p for p in sorted((KIT / "scenarios").rglob("*.yaml"))}
for name, path in LIBRARY.items():
    print(f"{path.parent.parent.name + '/' + path.parent.name:22s} {name}")

def try_scenario(name, undefended_first=False, attacker="static", attack_mode="static", show_trace=True):
    path = str(LIBRARY[name])
    if undefended_first:
        print(f"--- {name}: no defense (an attack scenario must report attack_success=True here) ---")
        sentinel("run", "--scenario", path, "--defense", "allow_all", "--model", WEIGHTS,
                 "--artifacts", str(ARTIFACTS), "--no-timeline")
    print(f"--- {name}: SENTINEL defense ---")
    sentinel("run", "--scenario", path, "--defense-url", DEFENSE_URL, "--model", WEIGHTS,
             "--attacker", attacker, "--attack-mode", attack_mode, "--artifacts", str(ARTIFACTS))
    dest = file_traces(f"try-{name}")
    if show_trace:
        for trace in sorted(dest.glob(f"{name}-*.jsonl")):
            if ".prev" not in trace.name:
                ours("sentinel_cli.py", "replay", str(trace))

try_scenario("soc_hostile_log_text", undefended_first=True)


In [ ]:
# The specification's own first example, and the one injection Qwen3-8B reliably falls for.
# Undefended first (the attack must succeed), then defended: this is the video's centre shot.
try_scenario("enterprise_poisoned_invoice", undefended_first=True)

## 7 · The evaluation

One pass per (seed, split, attacker): with the defaults that is four passes, and each pass runs
every scenario in its split exactly once. Each writes a
scorecard JSON into `results/`, and the per-scenario event logs land in `artifacts/` — those
JSONL files are what `sentinel replay` renders, and what the video and the report cite.

In [ ]:
def seed_config(seed: int) -> str:
    """The kit takes the run seed from a competition config, not a flag."""
    path = WORK / f"competition-s{seed}.yaml"
    path.write_text(f"run_seed: {seed}\n")
    return str(path)

for seed in SEEDS:
    for split, attacker, mode in RUNS:
        name = f"qwen3-8b-{split}-{attacker}-{mode}-s{seed}"
        print(f"\n{'=' * 72}\n{name}\n{'=' * 72}")
        sentinel("eval", split, "--defense-url", DEFENSE_URL, "--model", WEIGHTS,
                 "--attacker", attacker, "--attack-mode", mode, "--config", seed_config(seed),
                 "--artifacts", str(ARTIFACTS), "--output", str(RESULTS / f"{name}.json"))
        file_traces(name)


## 8 · Baselines under the same agent (optional)

Only meaningful if you have the quota: each is another full pass over the split.

In [ ]:
if RUN_CONTROL:
    for seed in SEEDS:
        for split in ("public", "validation"):
            name = f"qwen3-8b-{split}-allow_all-s{seed}"
            print(f"\n{'=' * 72}\ncontrol (no defense): {name}\n{'=' * 72}")
            sentinel("eval", split, "--defense", "allow_all", "--model", WEIGHTS,
                     "--config", seed_config(seed),
                     "--artifacts", str(ARTIFACTS), "--output", str(RESULTS / f"{name}.json"))

if RUN_BASELINES:
    for baseline in ("provenance", "heuristic_risk"):
        print(f"\n{'=' * 72}\nbaseline {baseline}\n{'=' * 72}")
        sentinel("eval", "public", "--defense", baseline, "--model", WEIGHTS,
                 "--artifacts", str(ARTIFACTS), "--output", str(RESULTS / f"qwen3-8b-public-{baseline}.json"))
else:
    print("kit baselines skipped (RUN_BASELINES = False)")


## 9 · Results

The table goes in `docs/OFFICIAL_HARNESS.md`. The failure list underneath it is the part to read
carefully: with a real agent a scenario can end in `model_error` or `max_steps`, which is the
agent failing the task rather than the defense blocking it, and the two must not be conflated.

In [ ]:
import collect_scorecards

cards = collect_scorecards.load(RESULTS)
report = collect_scorecards.report(cards)
(RESULTS / "RESULTS_QWEN3.md").write_text(report + "\n")
print(report)

# The observability layer over everything recorded above: one self-contained HTML file.
traces = sorted(str(p) for p in TRACES.rglob("*.jsonl") if ".prev" not in p.name)
if traces:
    ours("sentinel_cli.py", "dashboard", *traces, "--out", str(RESULTS / "dashboard_qwen3.html"))
else:
    print("no traces recorded: the defense service was never called")


In [ ]:
# One archive to download from the notebook's Output tab: scorecards, the dashboard, our traces,
# the harness event logs and the service log. Staged outside WORK so the zip cannot swallow itself.
stage = SCRATCH / "sentinel-qwen3-results"
shutil.rmtree(stage, ignore_errors=True)
for d in (RESULTS, TRACES, ARTIFACTS):
    shutil.copytree(d, stage / d.name)
archive = shutil.make_archive(str(WORK / "sentinel-qwen3-results"), "zip", root_dir=str(stage))
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")


## What to do with this

1. Put the table and the failure list into `docs/OFFICIAL_HARNESS.md`, alongside the mock-model
   numbers rather than replacing them — the gap between the two *is* a result, and says how much
   of the mock's perfect score came from the mock following a reference plan.
2. Declare the runtime in the technical report: precision, device map, that the weights are loaded
   once per process, and that the prompt, tool cards, decoding and step budget are the kit's own.
   `qwen_runtime.apply()` prints exactly that line.
3. Record the video from these artifacts: `sentinel replay artifacts/<group>/<run>.jsonl`.